In [8]:
import subprocess
import librosa
import numpy as np
import json
import os
import time
import re
import urllib.parse as urlparse

timestamps_json = '''
[{"t":0,"mix":0},{"t":7.786,"mix":1},{"t":15.305,"mix":2},{"t":22.389,"mix":3},{"t":29.34,"mix":4},{"t":36.417,"mix":5},{"t":42.772,"mix":6},{"t":50.295,"mix":7},{"t":57.414,"mix":8},{"t":63.97,"mix":9},{"t":71.727,"mix":10},{"t":79.517,"mix":11},{"t":86.591,"mix":12},{"t":93.396,"mix":13},{"t":100.838,"mix":14},{"t":108.195,"mix":15},{"t":114.912,"mix":16},{"t":122.8,"mix":17},{"t":128.854,"mix":18},{"t":135.18,"mix":19},{"t":141.412,"mix":20},{"t":147.997,"mix":21},{"t":155.764,"mix":22},{"t":159.156,"mix":23},{"t":163.26,"mix":24},{"t":169.719,"mix":25},{"t":179.35,"mix":26},{"t":185.797,"mix":27},{"t":186.897,"mix":28},{"t":188.06,"mix":29},{"t":189.226,"mix":30},{"t":190.465,"mix":31},{"t":191.469,"mix":32},{"t":192.691,"mix":33},{"t":193.74,"mix":34},{"t":194.964,"mix":35},{"t":196.217,"mix":36},{"t":197.387,"mix":37},{"t":198.581,"mix":38},{"t":199.721,"mix":39},{"t":200.737,"mix":40},{"t":201.927,"mix":41},{"t":203.135,"mix":42},{"t":204.401,"mix":43},{"t":205.726,"mix":44},{"t":206.83,"mix":45},{"t":207.95,"mix":46},{"t":209.13,"mix":47},{"t":210.343,"mix":48},{"t":211.523,"mix":49},{"t":212.713,"mix":50},{"t":213.913,"mix":51},{"t":215.079,"mix":52},{"t":216.235,"mix":53},{"t":217.422,"mix":54},{"t":218.595,"mix":55},{"t":219.777,"mix":56},{"t":220.963,"mix":57},{"t":222.298,"mix":58},{"t":223.462,"mix":59},{"t":224.576,"mix":60},{"t":225.656,"mix":61},{"t":226.866,"mix":62},{"t":228.558,"mix":63},{"t":230.015,"mix":64},{"t":231.247,"mix":65},{"t":232.482,"mix":66},{"t":233.678,"mix":67},{"t":234.8,"mix":68},{"t":235.938,"mix":69},{"t":237.033,"mix":70},{"t":238.412,"mix":71},{"t":239.68,"mix":72},{"t":240.878,"mix":73},{"t":241.987,"mix":74},{"t":243.149,"mix":75},{"t":244.369,"mix":76},{"t":245.771,"mix":77},{"t":247.179,"mix":78},{"t":249.033,"mix":79},{"t":250.237,"mix":80},{"t":251.528,"mix":81},{"t":252.775,"mix":82},{"t":253.903,"mix":83},{"t":255.54,"mix":84},{"t":256.999,"mix":85},{"t":258.15,"mix":86},{"t":259.286,"mix":87},{"t":260.456,"mix":88},{"t":261.506,"mix":89},{"t":262.725,"mix":90},{"t":264.034,"mix":91},{"t":265.177,"mix":92},{"t":266.319,"mix":93},{"t":267.247,"mix":94},{"t":268.225,"mix":95},{"t":269.477,"mix":96},{"t":270.525,"mix":97},{"t":271.65,"mix":98},{"t":272.788,"mix":99},{"t":273.744,"mix":100},{"t":275.004,"mix":101},{"t":276.737,"mix":102},{"t":277.911,"mix":103},{"t":279.238,"mix":104},{"t":280.427,"mix":105},{"t":281.653,"mix":106},{"t":282.889,"mix":107},{"t":283.987,"mix":108},{"t":285.734,"mix":109},{"t":286.978,"mix":110},{"t":288.298,"mix":111},{"t":289.408,"mix":112},{"t":290.613,"mix":113},{"t":291.884,"mix":114},{"t":292.924,"mix":115},{"t":294.425,"mix":116},{"t":295.519,"mix":117},{"t":296.754,"mix":118},{"t":297.976,"mix":119},{"t":299.243,"mix":120},{"t":300.437,"mix":121},{"t":301.686,"mix":122},{"t":302.899,"mix":123},{"t":304.205,"mix":124},{"t":305.364,"mix":125},{"t":306.612,"mix":126},{"t":307.778,"mix":127},{"t":308.894,"mix":128},{"t":310.209,"mix":129},{"t":311.714,"mix":130},{"t":312.884,"mix":131},{"t":314.023,"mix":132},{"t":315.167,"mix":133},{"t":316.44,"mix":134},{"t":317.654,"mix":135},{"t":318.674,"mix":136},{"t":320.266,"mix":137},{"t":321.611,"mix":138},{"t":322.955,"mix":139},{"t":324.217,"mix":140},{"t":325.385,"mix":141},{"t":326.733,"mix":142},{"t":328.088,"mix":143},{"t":329.334,"mix":144},{"t":330.55,"mix":145},{"t":331.736,"mix":146},{"t":332.965,"mix":147},{"t":334.11,"mix":148},{"t":335.408,"mix":149},{"t":336.965,"mix":150},{"t":338.063,"mix":151},{"t":339.387,"mix":152},{"t":340.544,"mix":153},{"t":341.685,"mix":154},{"t":342.851,"mix":155},{"t":343.874,"mix":156},{"t":344.898,"mix":157},{"t":346.004,"mix":158},{"t":347.096,"mix":159},{"t":348.281,"mix":160},{"t":349.527,"mix":161},{"t":350.718,"mix":162},{"t":351.866,"mix":163},{"t":353.017,"mix":164},{"t":354.267,"mix":165},{"t":355.514,"mix":166},{"t":356.772,"mix":167},{"t":357.905,"mix":168},{"t":359.166,"mix":169},{"t":360.354,"mix":170},{"t":361.642,"mix":171},{"t":362.908,"mix":172},{"t":364.11,"mix":173},{"t":365.343,"mix":174},{"t":366.794,"mix":175},{"t":368.223,"mix":176},{"t":369.451,"mix":177},{"t":370.833,"mix":178},{"t":372.017,"mix":179},{"t":373.369,"mix":180},{"t":374.495,"mix":181},{"t":376.249,"mix":182},{"t":377.938,"mix":183},{"t":379.317,"mix":184},{"t":380.511,"mix":185},{"t":381.589,"mix":186},{"t":382.873,"mix":187},{"t":384.124,"mix":188},{"t":385.356,"mix":189},{"t":386.433,"mix":190},{"t":387.563,"mix":191},{"t":388.831,"mix":192},{"t":390.065,"mix":193},{"t":391.197,"mix":194},{"t":392.312,"mix":195},{"t":393.767,"mix":196},{"t":395.163,"mix":197},{"t":396.331,"mix":198},{"t":397.655,"mix":199},{"t":398.961,"mix":200},{"t":400.298,"mix":201},{"t":402.331,"mix":202},{"t":404.402,"mix":203},{"t":405.665,"mix":204},{"t":406.85,"mix":205},{"t":408.056,"mix":206},{"t":409.189,"mix":207},{"t":410.914,"mix":208},{"t":412.357,"mix":209},{"t":413.575,"mix":210},{"t":414.893,"mix":211},{"t":416.128,"mix":212},{"t":417.359,"mix":213},{"t":418.826,"mix":214},{"t":419.937,"mix":215},{"t":421.304,"mix":216},{"t":423.154,"mix":217},{"t":425.232,"mix":218},{"t":427.47,"mix":219},{"t":429.695,"mix":220},{"t":431.072,"mix":221},{"t":433.526,"mix":222},{"t":443.098,"mix":223},{"t":444.169,"mix":224},{"t":448.692,"mix":225},{"t":452.405,"mix":226},{"t":456.988,"mix":227},{"t":461.324,"mix":228},{"t":465.683,"mix":229},{"t":469.689,"mix":230},{"t":474.083,"mix":231},{"t":478.411,"mix":232},{"t":483.099,"mix":233},{"t":487.02,"mix":234},{"t":491.432,"mix":235},{"t":495.019,"mix":236},{"t":498.73,"mix":237},{"t":502.864,"mix":238},{"t":507.259,"mix":239},{"t":510.827,"mix":240},{"t":515.083,"mix":241},{"t":518.423,"mix":223},{"t":519.084,"mix":224},{"t":523.51,"mix":225},{"t":527.358,"mix":226},{"t":531.916,"mix":227},{"t":536.357,"mix":228},{"t":541.165,"mix":229},{"t":545.126,"mix":230},{"t":549.192,"mix":231},{"t":553.445,"mix":232},{"t":557.928,"mix":233},{"t":561.778,"mix":234},{"t":566.086,"mix":235},{"t":569.673,"mix":236},{"t":573.578,"mix":237},{"t":577.721,"mix":238},{"t":582.327,"mix":239},{"t":586.136,"mix":240},{"t":590.315,"mix":241},{"t":593.606,"mix":242},{"t":594.412,"mix":243},{"t":598.771,"mix":244},{"t":602.833,"mix":245},{"t":607.234,"mix":246},{"t":611.487,"mix":247},{"t":615.76,"mix":248},{"t":620.254,"mix":249},{"t":625.02,"mix":250},{"t":628.549,"mix":251},{"t":632.951,"mix":252},{"t":636.531,"mix":253},{"t":640.895,"mix":254},{"t":644.745,"mix":255},{"t":649.238,"mix":256},{"t":653.703,"mix":257},{"t":658.057,"mix":258},{"t":662.031,"mix":259},{"t":666.008,"mix":260},{"t":670.071,"mix":242},{"t":670.88,"mix":243},{"t":675.153,"mix":244},{"t":679.209,"mix":245},{"t":683.884,"mix":246},{"t":687.921,"mix":247},{"t":691.975,"mix":248},{"t":696.461,"mix":249},{"t":701.043,"mix":250},{"t":705.142,"mix":251},{"t":710.123,"mix":252},{"t":713.98,"mix":253},{"t":717.934,"mix":254},{"t":721.829,"mix":255},{"t":726.519,"mix":256},{"t":730.974,"mix":257},{"t":735.102,"mix":258},{"t":738.996,"mix":259},{"t":743.357,"mix":260},{"t":754.128,"mix":261},{"t":754.856,"mix":262},{"t":757.095,"mix":263},{"t":759.795,"mix":264},{"t":762.244,"mix":265},{"t":765.187,"mix":266},{"t":767.821,"mix":267},{"t":770.677,"mix":268},{"t":773.352,"mix":269},{"t":775.73,"mix":270},{"t":778.59,"mix":271},{"t":781.494,"mix":272},{"t":784.149,"mix":273},{"t":786.452,"mix":261},{"t":787.132,"mix":262},{"t":789.485,"mix":263},{"t":792.235,"mix":264},{"t":794.851,"mix":265},{"t":797.942,"mix":266},{"t":800.544,"mix":267},{"t":803.28,"mix":268},{"t":806.058,"mix":269},{"t":808.774,"mix":270},{"t":811.593,"mix":271},{"t":814.322,"mix":272},{"t":816.89,"mix":273},{"t":819.459,"mix":274},{"t":820.027,"mix":275},{"t":822.839,"mix":276},{"t":825.817,"mix":277},{"t":828.855,"mix":278},{"t":831.912,"mix":279},{"t":834.641,"mix":280},{"t":837.759,"mix":281},{"t":840.784,"mix":282},{"t":843.628,"mix":283},{"t":846.748,"mix":284},{"t":849.597,"mix":285},{"t":852.649,"mix":286},{"t":856.246,"mix":274},{"t":856.815,"mix":275},{"t":859.489,"mix":276},{"t":862.466,"mix":277},{"t":865.461,"mix":278},{"t":868.381,"mix":279},{"t":871.065,"mix":280},{"t":874.381,"mix":281},{"t":877.441,"mix":282},{"t":880.181,"mix":283},{"t":883.112,"mix":284},{"t":886.004,"mix":285},{"t":889.365,"mix":286},{"t":900.025,"mix":287},{"t":906.371,"mix":288},{"t":911.146,"mix":289},{"t":916.338,"mix":290},{"t":921.897,"mix":291},{"t":927.543,"mix":292},{"t":932.47,"mix":293},{"t":937.866,"mix":294},{"t":942.864,"mix":287},{"t":947.845,"mix":288},{"t":953.041,"mix":289},{"t":958.501,"mix":290},{"t":964.442,"mix":291},{"t":969.539,"mix":292},{"t":974.495,"mix":293},{"t":979.505,"mix":294},{"t":983.96,"mix":295},{"t":988.332,"mix":296},{"t":993.67,"mix":297},{"t":999.183,"mix":298},{"t":1004.813,"mix":299},{"t":1010.175,"mix":300},{"t":1015.479,"mix":301},{"t":1020.765,"mix":302},{"t":1026.3,"mix":303},{"t":1032.081,"mix":304},{"t":1037.469,"mix":305},{"t":1042.885,"mix":306},{"t":1049.237,"mix":295},{"t":1054.473,"mix":296},{"t":1060.211,"mix":297},{"t":1066.334,"mix":298},{"t":1072.75,"mix":299},{"t":1078.491,"mix":300},{"t":1085.12,"mix":301},{"t":1090.533,"mix":302},{"t":1097.032,"mix":303},{"t":1103.023,"mix":304},{"t":1109.222,"mix":305},{"t":1116.479,"mix":306},{"t":1131.563,"mix":307},{"t":1133.688,"mix":308},{"t":1135.834,"mix":309},{"t":1137.935,"mix":310},{"t":1140.178,"mix":311},{"t":1142.378,"mix":312},{"t":1144.372,"mix":313},{"t":1146.384,"mix":314},{"t":1148.548,"mix":315},{"t":1150.496,"mix":316},{"t":1152.761,"mix":317},{"t":1155.017,"mix":318},{"t":1157.375,"mix":319},{"t":1158.709,"mix":307},{"t":1160.606,"mix":308},{"t":1162.927,"mix":309},{"t":1165.052,"mix":310},{"t":1167.195,"mix":311},{"t":1169.289,"mix":312},{"t":1171.358,"mix":313},{"t":1173.439,"mix":314},{"t":1175.508,"mix":315},{"t":1177.641,"mix":316},{"t":1179.839,"mix":317},{"t":1182.072,"mix":318},{"t":1184.474,"mix":319},{"t":1185.843,"mix":320},{"t":1187.584,"mix":321},{"t":1189.923,"mix":322},{"t":1192.245,"mix":323},{"t":1194.268,"mix":324},{"t":1196.417,"mix":325},{"t":1198.488,"mix":326},{"t":1200.667,"mix":327},{"t":1202.707,"mix":328},{"t":1204.953,"mix":329},{"t":1207.2,"mix":330},{"t":1209.483,"mix":331},{"t":1211.512,"mix":332},{"t":1214.669,"mix":333},{"t":1216.986,"mix":334},{"t":1219.227,"mix":335},{"t":1221.503,"mix":336},{"t":1224.029,"mix":337},{"t":1226.175,"mix":338},{"t":1228.794,"mix":339},{"t":1230.897,"mix":340},{"t":1233.365,"mix":341},{"t":1235.502,"mix":342},{"t":1237.982,"mix":343},{"t":1240.582,"mix":344},{"t":1241.941,"mix":320},{"t":1243.493,"mix":321},{"t":1245.843,"mix":322},{"t":1248.071,"mix":323},{"t":1250.114,"mix":324},{"t":1252.246,"mix":325},{"t":1254.27,"mix":326},{"t":1256.351,"mix":327},{"t":1258.424,"mix":328},{"t":1260.752,"mix":329},{"t":1263.159,"mix":330},{"t":1265.366,"mix":331},{"t":1267.41,"mix":332},{"t":1270.48,"mix":333},{"t":1272.709,"mix":334},{"t":1274.714,"mix":335},{"t":1277.043,"mix":336},{"t":1279.583,"mix":337},{"t":1281.837,"mix":338},{"t":1284.38,"mix":339},{"t":1286.396,"mix":340},{"t":1289.021,"mix":341},{"t":1291.314,"mix":342},{"t":1293.835,"mix":343},{"t":1296.763,"mix":344},{"t":1298.554,"mix":345},{"t":1299.631,"mix":346},{"t":1301.573,"mix":347},{"t":1303.305,"mix":348},{"t":1305.106,"mix":349},{"t":1306.232,"mix":345},{"t":1307.473,"mix":346},{"t":1309.459,"mix":347},{"t":1311.169,"mix":348},{"t":1312.974,"mix":349},{"t":1314.061,"mix":350},{"t":1315.076,"mix":351},{"t":1317.135,"mix":352},{"t":1319.203,"mix":353},{"t":1321.157,"mix":354},{"t":1323.305,"mix":355},{"t":1325.49,"mix":356},{"t":1327.227,"mix":357},{"t":1329.13,"mix":358},{"t":1330.95,"mix":359},{"t":1332.878,"mix":360},{"t":1334.675,"mix":361},{"t":1336.717,"mix":362},{"t":1338.73,"mix":363},{"t":1340.711,"mix":364},{"t":1342.566,"mix":365},{"t":1344.379,"mix":366},{"t":1346.038,"mix":367},{"t":1347.881,"mix":368},{"t":1349.094,"mix":350},{"t":1350.326,"mix":351},{"t":1352.397,"mix":352},{"t":1354.316,"mix":353},{"t":1356.239,"mix":354},{"t":1358.474,"mix":355},{"t":1360.545,"mix":356},{"t":1362.274,"mix":357},{"t":1364.229,"mix":358},{"t":1366.068,"mix":359},{"t":1368.086,"mix":360},{"t":1369.791,"mix":361},{"t":1371.921,"mix":362},{"t":1373.898,"mix":363},{"t":1376.116,"mix":364},{"t":1377.897,"mix":365},{"t":1379.539,"mix":366},{"t":1381.329,"mix":367},{"t":1383.271,"mix":368},{"t":1384.989,"mix":307},{"t":1386.911,"mix":308},{"t":1389.164,"mix":309},{"t":1391.252,"mix":310},{"t":1393.343,"mix":311},{"t":1395.679,"mix":312},{"t":1397.653,"mix":313},{"t":1399.79,"mix":314},{"t":1401.777,"mix":315},{"t":1403.87,"mix":316},{"t":1406.017,"mix":317},{"t":1408.233,"mix":318},{"t":1410.768,"mix":319},{"t":1412.201,"mix":320},{"t":1413.675,"mix":321},{"t":1416.156,"mix":322},{"t":1418.137,"mix":323},{"t":1420.289,"mix":324},{"t":1422.398,"mix":325},{"t":1424.509,"mix":326},{"t":1426.61,"mix":327},{"t":1428.697,"mix":328},{"t":1431.028,"mix":329},{"t":1433.354,"mix":330},{"t":1435.631,"mix":331},{"t":1437.684,"mix":332},{"t":1440.803,"mix":333},{"t":1442.928,"mix":334},{"t":1445.109,"mix":335},{"t":1447.304,"mix":336},{"t":1449.908,"mix":337},{"t":1452.109,"mix":338},{"t":1454.735,"mix":339},{"t":1456.756,"mix":340},{"t":1459.478,"mix":341},{"t":1461.673,"mix":342},{"t":1464.247,"mix":343},{"t":1467.205,"mix":344},{"t":1474.246,"mix":369},{"t":1475.12,"mix":370},{"t":1476.044,"mix":371},{"t":1476.974,"mix":372},{"t":1477.897,"mix":373},{"t":1478.804,"mix":374},{"t":1479.865,"mix":375},{"t":1480.858,"mix":376},{"t":1481.86,"mix":377},{"t":1482.787,"mix":378},{"t":1483.765,"mix":379},{"t":1484.704,"mix":380},{"t":1485.688,"mix":381},{"t":1486.409,"mix":382},{"t":1487.242,"mix":383},{"t":1488.215,"mix":384},{"t":1489.11,"mix":385},{"t":1490.041,"mix":386},{"t":1490.842,"mix":387},{"t":1491.754,"mix":388},{"t":1492.71,"mix":389},{"t":1493.473,"mix":390},{"t":1494.429,"mix":391},{"t":1495.439,"mix":392},{"t":1496.351,"mix":393},{"t":1497.02,"mix":369},{"t":1497.558,"mix":370},{"t":1498.567,"mix":371},{"t":1499.449,"mix":372},{"t":1500.366,"mix":373},{"t":1501.268,"mix":374},{"t":1502.201,"mix":375},{"t":1503.08,"mix":376},{"t":1504.088,"mix":377},{"t":1505.021,"mix":378},{"t":1506.074,"mix":379},{"t":1506.835,"mix":380},{"t":1507.763,"mix":381},{"t":1508.644,"mix":382},{"t":1509.635,"mix":383},{"t":1510.54,"mix":384},{"t":1511.426,"mix":385},{"t":1512.416,"mix":386},{"t":1513.201,"mix":387},{"t":1514.12,"mix":388},{"t":1515,"mix":389},{"t":1515.719,"mix":390},{"t":1516.717,"mix":391},{"t":1517.598,"mix":392},{"t":1518.643,"mix":393},{"t":1519.243,"mix":394},{"t":1519.557,"mix":395},{"t":1520.591,"mix":396},{"t":1521.538,"mix":397},{"t":1522.365,"mix":398},{"t":1523.397,"mix":399},{"t":1524.329,"mix":400},{"t":1525.299,"mix":401},{"t":1526.311,"mix":402},{"t":1527.084,"mix":403},{"t":1528.166,"mix":404},{"t":1529.19,"mix":405},{"t":1530.151,"mix":406},{"t":1531.105,"mix":407},{"t":1532.022,"mix":408},{"t":1532.985,"mix":409},{"t":1533.875,"mix":410},{"t":1534.929,"mix":411},{"t":1535.841,"mix":412},{"t":1536.782,"mix":413},{"t":1537.609,"mix":414},{"t":1538.572,"mix":415},{"t":1539.536,"mix":416},{"t":1540.47,"mix":417},{"t":1541.332,"mix":418},{"t":1542.384,"mix":419},{"t":1543.325,"mix":420},{"t":1544.242,"mix":421},{"t":1545.146,"mix":422},{"t":1546.059,"mix":423},{"t":1546.95,"mix":424},{"t":1547.853,"mix":425},{"t":1548.781,"mix":426},{"t":1549.705,"mix":427},{"t":1550.716,"mix":428},{"t":1551.678,"mix":429},{"t":1552.667,"mix":430},{"t":1553.591,"mix":431},{"t":1554.535,"mix":432},{"t":1555.478,"mix":433},{"t":1556.363,"mix":434},{"t":1557.262,"mix":435},{"t":1558.132,"mix":436},{"t":1558.896,"mix":437},{"t":1559.83,"mix":438},{"t":1560.792,"mix":439},{"t":1561.746,"mix":440},{"t":1562.742,"mix":441},{"t":1563.72,"mix":442},{"t":1564.338,"mix":394},{"t":1564.864,"mix":395},{"t":1565.746,"mix":396},{"t":1566.759,"mix":397},{"t":1567.635,"mix":398},{"t":1568.556,"mix":399},{"t":1569.629,"mix":400},{"t":1570.583,"mix":401},{"t":1571.576,"mix":402},{"t":1572.545,"mix":403},{"t":1573.477,"mix":404},{"t":1574.42,"mix":405},{"t":1575.352,"mix":406},{"t":1576.337,"mix":407},{"t":1577.393,"mix":408},{"t":1578.331,"mix":409},{"t":1579.257,"mix":410},{"t":1580.624,"mix":411},{"t":1581.7,"mix":412},{"t":1582.584,"mix":413},{"t":1583.454,"mix":414},{"t":1584.397,"mix":415},{"t":1585.427,"mix":416},{"t":1586.381,"mix":417},{"t":1587.26,"mix":418},{"t":1588.369,"mix":419},{"t":1589.303,"mix":420},{"t":1590.326,"mix":421},{"t":1591.219,"mix":422},{"t":1592.08,"mix":423},{"t":1592.98,"mix":424},{"t":1593.84,"mix":425},{"t":1594.685,"mix":426},{"t":1595.674,"mix":427},{"t":1596.714,"mix":428},{"t":1597.726,"mix":429},{"t":1598.623,"mix":430},{"t":1599.474,"mix":431},{"t":1600.47,"mix":432},{"t":1601.399,"mix":433},{"t":1602.383,"mix":434},{"t":1603.301,"mix":435},{"t":1604.234,"mix":436},{"t":1605.053,"mix":437},{"t":1606.025,"mix":438},{"t":1607.035,"mix":439},{"t":1608.038,"mix":440},{"t":1608.969,"mix":441},{"t":1610.676,"mix":442},{"t":1615.325,"mix":443}]
'''
timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)


def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            # Adjust the output path to have an m4a extension
            output_path = output_path.replace('.ogg', '.m4a')
            
            command = [
                'yt-dlp', '-x', '--audio-format', 'm4a',
                '-o', output_path, youtube_url
            ]
            subprocess.run(command, check=True)
            return output_path
        except subprocess.CalledProcessError as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

def load_audio_segment(audio_path, start_time=0, end_time=None, sr=11025):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
        duration = end_total_seconds - start_total_seconds
    else:
        # If end_time is None, calculate the full length of the audio
        full_duration = librosa.get_duration(filename=audio_path)
        duration = full_duration - start_total_seconds

    y, sr = librosa.load(audio_path, sr=sr, offset=start_total_seconds, duration=duration)
    return y, sr

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=WtDwMKkNeYk'
youtube_url2 = 'https://www.youtube.com/watch?v=d6DEkqEnh-U&t=19'

start_time1 = 16.58
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "27:11"
end_time2 = "24:37" #Use None for full length
full_duration2 = 0

# Download audio files
#audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.ogg')
if audio_path1:
    print('audio_path1 downloaded')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.ogg')
if audio_path2:
    print('audio_path2 downloaded')

# Convert to OGG
#ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
#convert_to_ogg(audio_path1.replace('.ogg', '.m4a'), ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2.replace('.ogg', '.m4a'), ogg_path2, start_time2, end_time2)

audio_path1 downloaded
[youtube] Extracting URL: https://www.youtube.com/watch?v=d6DEkqEnh-U&t=19
[youtube] d6DEkqEnh-U: Downloading webpage
[youtube] d6DEkqEnh-U: Downloading ios player API JSON
[youtube] d6DEkqEnh-U: Downloading player d2e656ee


         n = i_1QxBIldT6gt8QT ; player = https://www.youtube.com/s/player/d2e656ee/player_ias.vflset/en_US/base.js


[youtube] d6DEkqEnh-U: Downloading m3u8 information
[info] d6DEkqEnh-U: Downloading 1 format(s): 140
[download] Destination: youtube_audio2.m4a
[download] 100% of   22.45MiB in 00:00:03 at 5.66MiB/s     
[FixupM4a] Correcting container of "youtube_audio2.m4a"
[ExtractAudio] Not converting audio youtube_audio2.m4a; file is already in target format m4a
audio_path2 downloaded


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

'youtube_audio2.ogg'

In [2]:
import gc
from concurrent.futures import ThreadPoolExecutor

HOP_LENGTH = 128


def process_chunk(y_chunk, sr, hop_length):
    # Chroma feature extraction
    chroma = librosa.feature.chroma_stft(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
    # chroma = normalize_features(chroma)
    
    # Constant-Q Transform feature extraction
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
    # CQT = normalize_features(CQT)
    
    # Ensure the same length for both arrays
    min_length = min(chroma.shape[1], CQT.shape[1])
    chroma = chroma[:, :min_length]
    CQT = CQT[:, :min_length]
    
    # Combine features
    combined_chunk_features = np.vstack((chroma, CQT)).T
    
    return combined_chunk_features

def load_and_preprocess_audio_combined(audio_path, target_sr=44100, chunk_duration=10, hop_length=HOP_LENGTH):
    y, sr = librosa.load(audio_path, sr=target_sr)
    y = normalize_audio(y)  # Normalize the raw audio signal
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = max(1, num_chunks // 10)  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")

    def process_and_collect(i):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        chunk_features = process_chunk(y_chunk, sr, HOP_LENGTH)
        return chunk_features

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_and_collect, i) for i in range(num_chunks)]
        for i, future in enumerate(futures):
            combined_features.append(future.result())
            # Print progress
            if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
                print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")
            gc.collect()

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

# Normalize the raw audio signal
def normalize_audio(y, epsilon=1e-8):
    return (y - np.mean(y)) / (np.std(y) + epsilon)
    
def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format

S1, sr1 = load_and_preprocess_audio_combined(ogg_path1)

Total chunks: 162, Progress step: 16
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 40% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 99% of chunks
Processed 100% of chunks
Combined features shape: (556278, 96)


In [9]:
#making own step so no need to re-run audio1 for multiple recordings.

S2, sr2 = load_and_preprocess_audio_combined(ogg_path2)

Total chunks: 146, Progress step: 14
Processed 10% of chunks
Processed 19% of chunks
Processed 29% of chunks
Processed 38% of chunks
Processed 48% of chunks
Processed 58% of chunks
Processed 67% of chunks
Processed 77% of chunks
Processed 86% of chunks
Processed 96% of chunks
Processed 100% of chunks
Combined features shape: (502375, 96)


In [10]:
print(S1.shape,S2.shape)

(556278, 96) (502375, 96)


In [11]:
from fastdtw import fastdtw
#from dtaidistance import dtw
    
def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

#def constrained_dtw(S1, S2):
#    # Calculate the DTW distance with a window constraint
#    distance, paths = dtw.warping_paths(S1, S2, window=1000)
#    # Find the best path
#    path = dtw.best_path(paths)
#    return path

#warping_path = constrained_dtw(S1, S2)
warping_path = dynamic_time_warping_approx(S1, S2)

In [12]:
def adjust_timestamps(wp, timestamps, sr):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t']) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1)

In [13]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t'])
        ratios.append(current_ratio)
    return ratios

def format_number(number):
    return f"{number:.3f}"

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios)):  # Include the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios), i + neighbor_count + 1)  # Include the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.4 * neighborhood_average or adjusted_ratios[i] < 0.6 * neighborhood_average:
            duration = adjusted_timestamps[i + 1]['t'] - adjusted_timestamps[i]['t'] if i + 1 < len(adjusted_timestamps) else None
            if duration is not None:
                duration = format_number(duration)
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "duration": duration,
                "original_ratio": format_number(original_ratios[i]) if i < len(original_ratios) else "0.000",
                "adjusted_ratio": format_number(adjusted_ratios[i]),
                "average_neighbors": format_number(neighborhood_average)
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.40):
        difference = adjusted_t - previous_t
#        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")
print(youtube_url2)

def calculate_full_duration(audio_path, start_time, end_time):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
    else:
        # Calculate the full length of the audio if end_time is None
        full_duration = librosa.get_duration(path=audio_path)
        end_total_seconds = full_duration

    full_duration = end_total_seconds - start_total_seconds
    return full_duration

full_duration2 = calculate_full_duration(ogg_path2, start_time2, end_time2)

#Flag anything that exceeds 40% difference compared to neighboring measures.
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, full_duration2)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Duration: {ft['duration']}, Original: {ft['original_ratio']}, Neighbors' Avg.: {ft['average_neighbors']}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
os.remove(audio_path2)
#os.remove(ogg_path1)
os.remove(ogg_path2)

[{"t": 0.0, "mix": 0}, {"t": 7.088, "mix": 1}, {"t": 12.963, "mix": 2}, {"t": 18.689, "mix": 3}, {"t": 23.864, "mix": 4}, {"t": 29.376, "mix": 5}, {"t": 34.92, "mix": 6}, {"t": 40.42, "mix": 7}, {"t": 45.575, "mix": 8}, {"t": 51.151, "mix": 9}, {"t": 56.291, "mix": 10}, {"t": 61.658, "mix": 11}, {"t": 67.1, "mix": 12}, {"t": 72.269, "mix": 13}, {"t": 77.915, "mix": 14}, {"t": 83.345, "mix": 15}, {"t": 88.584, "mix": 16}, {"t": 93.989, "mix": 17}, {"t": 99.59, "mix": 18}, {"t": 104.893, "mix": 19}, {"t": 109.973, "mix": 20}, {"t": 115.423, "mix": 21}, {"t": 121.853, "mix": 22}, {"t": 125.179, "mix": 23}, {"t": 129.292, "mix": 24}, {"t": 133.57, "mix": 25}, {"t": 142.144, "mix": 26}, {"t": 147.885, "mix": 27}, {"t": 149.153, "mix": 28}, {"t": 150.48, "mix": 29}, {"t": 151.722, "mix": 30}, {"t": 153.049, "mix": 31}, {"t": 154.291, "mix": 32}, {"t": 155.687, "mix": 33}, {"t": 156.761, "mix": 34}, {"t": 158.012, "mix": 35}, {"t": 159.483, "mix": 36}, {"t": 160.737, "mix": 37}, {"t": 162.009